# Using ollama

In this notebook, we use ollama to generate a fun fact. This notebook assumes the following:
- you have ollama installed (somewhere) and it is listening on the default port 11434
- you have a proxy in front of ollama that enforces the use of an API key, and this proxy is listening on port 11435, on the same host where ollama lives
- in your .env file, you have OLLAMA_HOST and OLLAMA_API_KEY configured. The host can be either host name or IP address, but without port information or http:// prefix.

# Setting up an nginx proxy to require an API key

We can either setup the proxy so the API key is expected in the `X-API-KEY` header, or so it is expected as a `Bearer` token in the `Authorization` header.
Then OpenAI API uses the latter version. For the first version, we need to pass in an extra header with each request with the API key.

First of all, we need to create folders to store the config and log files:
```bash
mkdir -p $HOME/docker-storage/nginx-ollama/{conf,logs,html}

# for the nginx image
sudo chown -R 101:$GROUPS $HOME/docker-storage/nginx-ollama/logs

# for the nginx-openresty image that uses lua scripts to log requests/responses
sudo chown -R 65534:$GROUPS $HOME/docker-storage/nginx-ollama/logs
```

For the proxy configs, fill in the `<OLLAMA IP>` placeholder and update the list of accepted API keys!

## nginx proxy config to require API key in the X-API-KEY header

nginx.conf
```
# --------------------------------------------------------------
# Global settings
# --------------------------------------------------------------
worker_processes  auto;
error_log        /var/log/nginx/error.log warn;

events {
    worker_connections  1024;
}

# --------------------------------------------------------------
# HTTP block
# --------------------------------------------------------------
http {
    # Include mime types (optional)
    include       mime.types;
    default_type  application/octet-stream;

    # ----------------------------------------------------------
    # Define the set of valid API keys
    # ----------------------------------------------------------
    # Keys are stored in a map:  <api-key>  -> 1 (valid) / 0 (invalid)
    # Add or remove entries here as needed.
    #
    # Example keys (replace with your real secrets):
    #   abcdef1234567890
    #   1122334455667788
    #
    # NOTE: keep this file private – treat it like a secret.
    # ----------------------------------------------------------
    map $http_x_api_key $key_valid {
        default                         0;          # reject unknown keys
        "abcdef1234567890"              1;
        "1122334455667788"              1;
    }

    # ----------------------------------------------------------
    # Choose a log format that includes the API key (if any)
    # ----------------------------------------------------------
    log_format  api_log  '$remote_addr - [$time_local] '
                         '"$request" $status $body_bytes_sent '
                         '"$http_user_agent" '
                         'api_key=$http_x_api_key '
                         'upstream_response_time=$upstream_response_time';

    # ----------------------------------------------------------
    # Upstream definition – points to the local Ollama service.
    # Using host.docker.internal lets the container reach the host’s
    # localhost (127.0.0.1) on macOS/Windows. On Linux you can use
    # the host network mode or the host’s IP address.
    # ----------------------------------------------------------
    upstream ollama_backend {
        server <OLLAMA IP>:11434;
    }

    # ----------------------------------------------------------
    # Server block (HTTPS recommended – replace with your certs)
    # ----------------------------------------------------------
    server {
        listen 80;                     # change to 443 + ssl for TLS
        server_name _;                 # catch‑all

        # ------------------------------------------------------------------
        # Access control – reject requests without a valid API key
        # ------------------------------------------------------------------
        if ($key_valid = 0) {
            return 401 "Invalid or missing API key";
        }

        # ------------------------------------------------------------------
        # Conditional logging – separate log file per API key
        # ------------------------------------------------------------------
        # If the key matches a known value, use its dedicated log.
        # Otherwise fall back to a generic log (should never happen because
        # the block above already rejected invalid keys).
        #
        # Note: the variable $log_file is created on‑the‑fly.
        # ------------------------------------------------------------------
        set $log_file "../../var/log/nginx/logs/access_default.log";

        if ($http_x_api_key = "abcdef1234567890") {
            set $log_file "../../var/log/nginx/logs/access_key1.log";
        }
        if ($http_x_api_key = "1122334455667788") {
            set $log_file "../../var/log/nginx/logs/access_key2.log";
        }

        access_log  $log_file  api_log;

        # ------------------------------------------------------------------
        # Proxy everything to Ollama
        # ------------------------------------------------------------------
        location / {
            proxy_pass http://ollama_backend;
            proxy_set_header Host $host;
            proxy_set_header X-Real-IP $remote_addr;
            proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
            proxy_set_header X-Forwarded-Proto $scheme;

            # Preserve the original API key for the backend (optional)
            proxy_set_header X-API-Key $http_x_api_key;
        }
    }
}
```

Start:
```bash
BASE=$HOME/docker-storage/nginx-ollama; docker run -d --restart=unless-stopped --name nginx-ollama-proxy -p 11435:80 -v $BASE/conf/nginx.conf:/etc/nginx/nginx.conf:ro -v $BASE/logs:/var/log/nginx/logs -v $BASE/html:/etc/nginx/html:ro nginx:1.28.0
```

## Passing in the API key in the X-API-KEY header and log requests/responses

For each API key, two folders need to be created inside the `logs/bodies` folder, one prefixed with `req_` and one with `resp_`.

nginx-openresty.conf
```
# --------------------------------------------------------------
# Global settings
# --------------------------------------------------------------
worker_processes  auto;
error_log        /var/log/nginx/error.log warn;

events {
    worker_connections  1024;
}

# --------------------------------------------------------------
# HTTP block
# --------------------------------------------------------------
http {
    # Include mime types (optional)
    #include       mime.types;
    default_type  application/octet-stream;

    # ----------------------------------------------------------
    # Define the set of valid API keys
    # ----------------------------------------------------------
    # Keys are stored in a map:  <api-key>  -> 1 (valid) / 0 (invalid)
    # Add or remove entries here as needed.
    #
    # Example keys (replace with your real secrets):
    #   abcdef1234567890
    #   1122334455667788
    #
    # NOTE: keep this file private – treat it like a secret.
    # ----------------------------------------------------------
    map $http_x_api_key $key_valid {
        default                         0;          # reject unknown keys
        "abcdef1234567890"              1;
        "1122334455667788"              1;
    }

    # ----------------------------------------------------------
    # Choose a log format that includes the API key (if any)
    # ----------------------------------------------------------
    log_format  api_log  '$remote_addr - [$time_local] '
                         '"$request" $status $body_bytes_sent '
                         '"$http_user_agent" '
                         'api_key=$http_x_api_key '
                         'upstream_response_time=$upstream_response_time';

    # ----------------------------------------------------------
    # Upstream definition – points to the local Ollama service.
    # Using host.docker.internal lets the container reach the host’s
    # localhost (127.0.0.1) on macOS/Windows. On Linux you can use
    # the host network mode or the host’s IP address.
    # ----------------------------------------------------------
    upstream ollama_backend {
        server <OLLAMA IP>:11434;
    }

    # ----------------------------------------------------------
    # Server block (HTTPS recommended – replace with your certs)
    # ----------------------------------------------------------
    server {
        listen 80;                     # change to 443 + ssl for TLS
        server_name _;                 # catch‑all

        # -------------------------------------------------
        #  Increase timeouts for the Ollama proxy
        # -------------------------------------------------
        # 1. Connection to Ollama
        #proxy_connect_timeout 30s;      # how long to wait for TCP handshake
        # 2. Sending request body to Ollama
        #proxy_send_timeout    5m;       # generous for large POST payloads
        # 3. Waiting for Ollama’s response (the most important one)
        proxy_read_timeout    10m;      # allow up to 10 minutes for a generation

        # Optional: tighten client‑side limits (helps with very slow callers)
        #client_body_timeout   5m;
        #keepalive_timeout     2m;

        # ------------------------------------------------------------------
        # Access control – reject requests without a valid API key
        # ------------------------------------------------------------------
        if ($key_valid = 0) {
            return 401 "Invalid or missing API key";
        }

        # ------------------------------------------------------------------
        # Conditional logging – separate log file per API key
        # ------------------------------------------------------------------
        # If the key matches a known value, use its dedicated log.
        # Otherwise fall back to a generic log (should never happen because
        # the block above already rejected invalid keys).
        #
        # Note: the variable $log_file is created on‑the‑fly.
        # ------------------------------------------------------------------
        set $log_file "../../../../var/log/nginx/logs/access_default.log";

        if ($key_valid = 1) {
            set $log_file "../../../../var/log/nginx/logs/access_$http_x_api_key.log";
        }

        access_log  $log_file  api_log;

        # ------------------------------------------------------
        #  *** Lua handlers ***
        # ------------------------------------------------------
        #  1. Capture the request body (only for POST/PUT/PATCH)
        #  2. After the upstream finishes, capture the response body
        # ------------------------------------------------------
        lua_need_request_body on;   # make $request_body available

        # Store request body to a file named by timestamp + API key
        set $req_body_path "../../../../var/log/nginx/logs/bodies/req_$http_x_api_key";
        set $resp_body_path "../../../../var/log/nginx/logs/bodies/resp_$http_x_api_key";

        # ------------------------------------------------------------------
        #  Phase: rewrite → run before proxy_pass
        # ------------------------------------------------------------------
        rewrite_by_lua_block {
            local api_key = ngx.var.http_x_api_key or "unknown"
            local method   = ngx.req.get_method()
            if method == "POST" or method == "PUT" or method == "PATCH" then
                ngx.req.read_body()
                local data = ngx.req.get_body_data()
                if data then
                    local ts   = ngx.time()
                    local fname = string.format("%s/%d_%s.json", ngx.var.req_body_path, ts, api_key)
                    local f, err = io.open(fname, "w")
                    if f then
                        f:write(data)
                        f:close()
                    else
                        ngx.log(ngx.ERR, "cannot write request body: ", err)
                    end
                end
            end
        }

        # ------------------------------------------------------------------
        # Proxy everything to Ollama
        # ------------------------------------------------------------------
        location / {
            proxy_pass http://ollama_backend;
            proxy_set_header Host $host;
            proxy_set_header X-Real-IP $remote_addr;
            proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
            proxy_set_header X-Forwarded-Proto $scheme;

            # Preserve the original API key for the backend (optional)
            proxy_set_header X-API-Key $http_x_api_key;

            # --------------------------------------------------------------
            #  Capture the upstream response body
            # --------------------------------------------------------------
            proxy_buffering off;                 # needed for body capture
            body_filter_by_lua_block {
                -- This runs for every chunk of the response.
                -- We'll accumulate the whole body in a per‑request table.
                local ctx = ngx.ctx
                if not ctx.resp_body then
                    ctx.resp_body = {}
                end
                local chunk = ngx.arg[1]
                if chunk ~= "" then
                    table.insert(ctx.resp_body, chunk)
                end

                -- When the last chunk arrives, write the full body to disk.
                if ngx.arg[2] then
                    local full_body = table.concat(ctx.resp_body)
                    local api_key = ngx.var.http_x_api_key or "unknown"
                    local ts = ngx.time()
                    local fname = string.format("%s/%d_%s.json",
                        ngx.var.resp_body_path, ts, api_key)

                    local f, err = io.open(fname, "w")
                    if f then
                        f:write(full_body)
                        f:close()
                    else
                        ngx.log(ngx.ERR, "cannot write response body: ", err)
                    end
                end
                -- pass the chunk downstream unchanged
                ngx.arg[1] = chunk
            }
        }
    }
}
```

Start:
```bash
BASE=$HOME/docker-storage/nginx-ollama; docker run -d --restart=unless-stopped --name nginx-ollama-proxy -p 11435:80 -v $BASE/conf/nginx-openresty.conf:/etc/nginx/nginx.conf:ro -v $BASE/logs:/var/log/nginx/logs -v $BASE/html:/etc/nginx/html:ro openresty/openresty:1.27.1.2-5-alpine nginx -c /etc/nginx/nginx.conf -g "daemon off;"
```


## Passing in the API key in the Authorization header as a Bearer token and log requests/responses

For each API key, two folders need to be created inside the `logs/bodies` folder, one prefixed with `req_` and one with `resp_`.

nginx-openresty-bearer-token.conf
```
# --------------------------------------------------------------
# Global settings
# --------------------------------------------------------------
worker_processes  auto;
error_log        /var/log/nginx/error.log warn;

events {
    worker_connections  1024;
}

# --------------------------------------------------------------
# HTTP block
# --------------------------------------------------------------
http {
    # Include mime types (optional)
    #include       mime.types;
    default_type  application/octet-stream;

    # ----------------------------------------------------------
    # Define the set of valid API keys
    # ----------------------------------------------------------
    # Keys are stored in a map:  <api-key>  -> 1 (valid) / 0 (invalid)
    # Add or remove entries here as needed.
    #
    # Example keys (replace with your real secrets):
    #   abcdef1234567890
    #   1122334455667788
    #
    # NOTE: keep this file private – treat it like a secret.
    # ----------------------------------------------------------
    map $api_key $key_valid {
        default                         0;          # reject unknown keys
        "abcdef1234567890"              1;
        "1122334455667788"              1;
    }

    # ----------------------------------------------------------
    # Choose a log format that includes the API key (if any)
    # ----------------------------------------------------------
    log_format  api_log  '$remote_addr - [$time_local] '
                         '"$request" $status $body_bytes_sent '
                         '"$http_user_agent" '
                         'api_key=$api_key '
                         'upstream_response_time=$upstream_response_time';

    # ----------------------------------------------------------
    # Upstream definition – points to the local Ollama service.
    # Using host.docker.internal lets the container reach the host’s
    # localhost (127.0.0.1) on macOS/Windows. On Linux you can use
    # the host network mode or the host’s IP address.
    # ----------------------------------------------------------
    upstream ollama_backend {
        server <OLLAMA IP>:11434;
    }

    # ----------------------------------------------------------
    # Server block (HTTPS recommended – replace with your certs)
    # ----------------------------------------------------------
    server {
        listen 80;                     # change to 443 + ssl for TLS
        server_name _;                 # catch‑all

        # ------------------------------------------------------------------
        # Strip "Bearer " from the Authorization header
        # ------------------------------------------------------------------
        # $http_authorization contains the whole header value, e.g.
        #   "Bearer abcdef1234567890"
        # The regex captures the token part (group 1) and discards the prefix.
        #
        # If the header is missing or does not start with "Bearer ",
        # $api_key will be an empty string.
        # ------------------------------------------------------------------
        set $api_key "";
        if ($http_authorization ~* ^Bearer\s+(.+)$) {
            set $api_key $1;
        }

        # -------------------------------------------------
        #  Increase timeouts for the Ollama proxy
        # -------------------------------------------------
        # 1. Connection to Ollama
        #proxy_connect_timeout 30s;      # how long to wait for TCP handshake
        # 2. Sending request body to Ollama
        #proxy_send_timeout    5m;       # generous for large POST payloads
        # 3. Waiting for Ollama’s response (the most important one)
        proxy_read_timeout    10m;      # allow up to 10 minutes for a generation

        # Optional: tighten client‑side limits (helps with very slow callers)
        #client_body_timeout   5m;
        #keepalive_timeout     2m;

        # ------------------------------------------------------------------
        # Access control – reject requests without a valid API key
        # ------------------------------------------------------------------
        if ($key_valid = 0) {
            return 401 "Invalid or missing API key";
        }

        # ------------------------------------------------------------------
        # Conditional logging – separate log file per API key
        # ------------------------------------------------------------------
        # If the key matches a known value, use its dedicated log.
        # Otherwise fall back to a generic log (should never happen because
        # the block above already rejected invalid keys).
        #
        # Note: the variable $log_file is created on‑the‑fly.
        # ------------------------------------------------------------------
        set $log_file "../../../../var/log/nginx/logs/access_default.log";

        if ($key_valid = 1) {
            set $log_file "../../../../var/log/nginx/logs/access_$api_key.log";
        }

        access_log  $log_file  api_log;

        # ------------------------------------------------------
        #  *** Lua handlers ***
        # ------------------------------------------------------
        #  1. Capture the request body (only for POST/PUT/PATCH)
        #  2. After the upstream finishes, capture the response body
        # ------------------------------------------------------
        lua_need_request_body on;   # make $request_body available

        # Store request body to a file named by timestamp + API key
        set $req_body_path "../../../../var/log/nginx/logs/bodies/req_$api_key";
        set $resp_body_path "../../../../var/log/nginx/logs/bodies/resp_$api_key";

        # ------------------------------------------------------------------
        #  Phase: rewrite → run before proxy_pass
        # ------------------------------------------------------------------
        rewrite_by_lua_block {
            local api_key = ngx.var.api_key or "unknown"
            local method   = ngx.req.get_method()
            if method == "POST" or method == "PUT" or method == "PATCH" then
                ngx.req.read_body()
                local data = ngx.req.get_body_data()
                if data then
                    local ts   = ngx.time()
                    local fname = string.format("%s/%d_%s.json", ngx.var.req_body_path, ts, api_key)
                    local f, err = io.open(fname, "w")
                    if f then
                        f:write(data)
                        f:close()
                    else
                        ngx.log(ngx.ERR, "cannot write request body: ", err)
                    end
                end
            end
        }

        # ------------------------------------------------------------------
        # Proxy everything to Ollama
        # ------------------------------------------------------------------
        location / {
            proxy_pass http://ollama_backend;
            proxy_set_header Host $host;
            proxy_set_header X-Real-IP $remote_addr;
            proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
            proxy_set_header X-Forwarded-Proto $scheme;

            # Preserve the original API key for the backend (optional)
            proxy_set_header X-API-Key $http_x_api_key;

            # --------------------------------------------------------------
            #  Capture the upstream response body
            # --------------------------------------------------------------
            proxy_buffering off;                 # needed for body capture
            body_filter_by_lua_block {
                -- This runs for every chunk of the response.
                -- We'll accumulate the whole body in a per‑request table.
                local ctx = ngx.ctx
                if not ctx.resp_body then
                    ctx.resp_body = {}
                end
                local chunk = ngx.arg[1]
                if chunk ~= "" then
                    table.insert(ctx.resp_body, chunk)
                end

                -- When the last chunk arrives, write the full body to disk.
                if ngx.arg[2] then
                    local full_body = table.concat(ctx.resp_body)
                    local api_key = ngx.var.api_key or "unknown"
                    local ts = ngx.time()
                    local fname = string.format("%s/%d_%s.json",
                        ngx.var.resp_body_path, ts, api_key)

                    local f, err = io.open(fname, "w")
                    if f then
                        f:write(full_body)
                        f:close()
                    else
                        ngx.log(ngx.ERR, "cannot write response body: ", err)
                    end
                end
                -- pass the chunk downstream unchanged
                ngx.arg[1] = chunk
            }
        }
    }
}
```

Start:
```bash
BASE=$HOME/docker-storage/nginx-ollama; docker run -d --restart=unless-stopped --name nginx-ollama-proxy -p 11435:80 -v $BASE/conf/nginx-openresty-bearer-token.conf:/etc/nginx/nginx.conf:ro -v $BASE/logs:/var/log/nginx/logs -v $BASE/html:/etc/nginx/html:ro openresty/openresty:1.27.1.2-5-alpine nginx -c /etc/nginx/nginx.conf -g "daemon off;"
```

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

OLLAMA_HOST = os.getenv('OLLAMA_HOST')
OLLAMA_API_KEY = os.getenv('OLLAMA_API_KEY')


In [12]:
# Retrieve what models are currently available for ollama
import requests
from IPython.display import display, Markdown

OLLAMA_BASE_URL = f"http://{OLLAMA_HOST}:11434/v1"
response = requests.get(f"{OLLAMA_BASE_URL}/models")
models = [ "- " + m['id'] for m in response.json()['data']]
display(Markdown('\n'.join(models)))

- deepseek-r1:7b-qwen-distill-q8_0
- jobautomation/OpenEuroLLM-Hungarian:latest
- llama3.1:8b-instruct-q6_K
- mannix/llama3.1-8b-lexi:q6_k
- mannix/llamax3-8b-alpaca:q6_k
- qwen2.5:14b-instruct-q6_K

In [15]:
# Run ollama from self-hosted server without API key
from openai import OpenAI

OLLAMA_BASE_URL = f"http://{OLLAMA_HOST}:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [16]:
# Get a fun fact

#response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])
response = ollama.chat.completions.create(model="llama3.1:8b-instruct-q6_K", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Here\'s one:\n\n**There is a species of jellyfish that is immortal!**\n\nThe Turritopsis dohrnii, also known as the "immortal jellyfish," is a type of jellyfish that can transform its body into a younger state through a process called transdifferentiation. This means that it can essentially revert back to its polyp stage, which is the juvenile form of a jellyfish, and then grow back into an adult again. This process can be repeated indefinitely, making the Turritopsis dohrnii theoretically immortal!\n\nIsn\'t that mind-blowing?'

In [21]:
# Run ollama from self-hosted server with nginx in front of it and API key requirement
from openai import OpenAI

OLLAMA_BASE_URL = f"http://{OLLAMA_HOST}:11435/v1"

#ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=f"{OLLAMA_API_KEY}")

In [23]:
# Get a fun fact

#response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])
#response = ollama.chat.completions.create(model="llama3.1:8b-instruct-q6_K", messages=[{"role": "user", "content": "Tell me a fun fact"}], extra_headers={"X-API-KEY": f"{OLLAMA_API_KEY}"})
response = ollama.chat.completions.create(model="llama3.1:8b-instruct-q6_K", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Here\'s one:\n\n**Did you know that there is a type of jellyfish that is immortal?**\n\nThe Turritopsis dohrnii, also known as the "immortal jellyfish," is a species of jellyfish that can transform its body into a younger state through a process called transdifferentiation. This means that it can essentially revert back to its polyp stage, which is the juvenile form of a jellyfish, and then grow back into an adult again. This process can be repeated indefinitely, making the Turritopsis dohrnii theoretically immortal!\n\nIsn\'t that just mind-blowing?'